In [1]:
import numpy as np
import torch
from pathlib import Path

processed_dir = Path("../data/processed")

X_train = np.load(processed_dir / "X_train_scaled.npy")
X_test = np.load(processed_dir / "X_test_scaled.npy")
y_train = np.load(processed_dir / "y_train.npy")
y_test = np.load(processed_dir / "y_test.npy")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

X_train: (175341, 194)
X_test: (82332, 194)
Device: cuda
GPU: NVIDIA GeForce RTX 5070


In [2]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)

X_train_tensor: torch.Size([175341, 194])
y_train_tensor: torch.Size([175341, 1])
X_test_tensor: torch.Size([82332, 194])
y_test_tensor: torch.Size([82332, 1])


In [3]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 1024

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 172
Testing batches: 81


In [4]:
import torch.nn as nn

class IntrusionDetector(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

model = IntrusionDetector(X_train.shape[1]).to(device)

print(model)
print("\nModel device:", next(model.parameters()).device)

IntrusionDetector(
  (network): Sequential(
    (0): Linear(in_features=194, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

Model device: cuda:0


In [5]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam


In [6]:
import time

epochs = 10

start_time = time.time()

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    print(
        f"Epoch {epoch + 1:02d}/{epochs} "
        f"- Loss: {epoch_loss:.4f} "
        f"- Accuracy: {epoch_accuracy:.4f}"
    )

elapsed = time.time() - start_time

print(f"\nTraining complete in {elapsed:.2f} seconds")

Epoch 01/10 - Loss: 0.2093 - Accuracy: 0.9073
Epoch 02/10 - Loss: 0.1276 - Accuracy: 0.9385
Epoch 03/10 - Loss: 0.1230 - Accuracy: 0.9395
Epoch 04/10 - Loss: 0.1205 - Accuracy: 0.9407
Epoch 05/10 - Loss: 0.1192 - Accuracy: 0.9416
Epoch 06/10 - Loss: 0.1186 - Accuracy: 0.9417
Epoch 07/10 - Loss: 0.1172 - Accuracy: 0.9419
Epoch 08/10 - Loss: 0.1165 - Accuracy: 0.9426
Epoch 09/10 - Loss: 0.1155 - Accuracy: 0.9430
Epoch 10/10 - Loss: 0.1146 - Accuracy: 0.9439

Training complete in 19.06 seconds


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

model.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())
        all_labels.append(y_batch)

y_prob_nn = torch.cat(all_probs).numpy().ravel()
y_pred_nn = torch.cat(all_preds).numpy().ravel()
y_true_nn = torch.cat(all_labels).numpy().ravel()

print("Accuracy :", accuracy_score(y_true_nn, y_pred_nn))
print("Precision:", precision_score(y_true_nn, y_pred_nn))
print("Recall   :", recall_score(y_true_nn, y_pred_nn))
print("F1 Score :", f1_score(y_true_nn, y_pred_nn))
print("ROC-AUC  :", roc_auc_score(y_true_nn, y_prob_nn))

print("\nClassification Report:")
print(classification_report(y_true_nn, y_pred_nn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_nn, y_pred_nn))

Accuracy : 0.8497789437885633
Precision: 0.7964495125723947
Recall   : 0.9768154945733698
F1 Score : 0.877459625483008
ROC-AUC  : 0.9741308538088959

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.69      0.81     37000
         1.0       0.80      0.98      0.88     45332

    accuracy                           0.85     82332
   macro avg       0.88      0.84      0.84     82332
weighted avg       0.87      0.85      0.85     82332


Confusion Matrix:
[[25683 11317]
 [ 1051 44281]]


In [8]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    model.state_dict(),
    models_dir / "pytorch_intrusion_detector.pth"
)

print("Saved:", models_dir / "pytorch_intrusion_detector.pth")

Saved: ..\models\pytorch_intrusion_detector.pth


In [9]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["XGBoost", "PyTorch Neural Network"],
    "Accuracy": [0.872953, 0.849779],
    "Precision": [0.821781, 0.796450],
    "Recall": [0.982286, 0.976815],
    "F1": [0.894893, 0.877460],
    "ROC_AUC": [0.983114, 0.974131]
})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,XGBoost,0.872953,0.821781,0.982286,0.894893,0.983114
1,PyTorch Neural Network,0.849779,0.796450,0.976815,0.877460,0.974131
